In [1]:
import torch

print("PyTorch Version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU Device Name:", torch.cuda.get_device_name(0))
    DEVICE = torch.device("cuda")
else:
    print("WARNING: GPU not detected. Running on CPU.")
    DEVICE = torch.device("cpu")

PyTorch Version: 2.5.1+cu121
CUDA Available: True
GPU Device Name: NVIDIA GeForce RTX 3050 Laptop GPU


In [2]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from PIL import Image
import pandas as pd
from tqdm.notebook import tqdm
from sklearn.metrics import accuracy_score, classification_report

In [3]:
# Augmentation for training, simple resizing/normalization for validation
train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(20),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [4]:
val_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [5]:
class APTOSDataset(Dataset):
    def __init__(self, df, img_dir, transform=None):
        self.df = df.reset_index(drop=True)
        self.img_dir = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_id = str(row['id_code']).strip()
        
        img_path = None
        for ext in ['', '.png', '.jpg', '.jpeg']:
            possible_path = os.path.join(self.img_dir, img_id + ext)
            if os.path.exists(possible_path):
                img_path = possible_path
                break
                
        if img_path is None:
            raise FileNotFoundError(f"Image {img_id} not found in {self.img_dir}")

        image = Image.open(img_path).convert('RGB')
        label = int(row['diagnosis'])

        if self.transform:
            image = self.transform(image)

        return image, label

In [6]:
DATA_DIR = os.path.join("..", "aptos")
TRAIN_CSV = os.path.join(DATA_DIR, "train_1.csv")
VAL_CSV = os.path.join(DATA_DIR, "valid.csv")
TRAIN_IMG_DIR = os.path.join(DATA_DIR, "train_images")
VAL_IMG_DIR = os.path.join(DATA_DIR, "val_images")

BATCH_SIZE = 16  # Optimal for RTX 3050 (4GB VRAM)

In [7]:
train_df = pd.read_csv(TRAIN_CSV)
val_df = pd.read_csv(VAL_CSV)

print(f"Loaded {len(train_df)} training and {len(val_df)} validation rows.")

Loaded 2930 training and 366 validation rows.


In [8]:
train_dataset = APTOSDataset(train_df, TRAIN_IMG_DIR, transform=train_transforms)
val_dataset = APTOSDataset(val_df, VAL_IMG_DIR, transform=val_transforms)

In [9]:
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)

In [10]:
# Load pre-trained ResNet-50
model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)

In [11]:
# Adjust final layer for 5 Diabetic Retinopathy classes
in_features = model.fc.in_features
model.fc = nn.Sequential(
    nn.Dropout(0.3),
    nn.Linear(in_features, 5)
)

In [12]:
model = model.to(DEVICE)

In [13]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-2)

In [14]:
# Enable AMP (Automatic Mixed Precision) for fast processing on RTX 3050
scaler = torch.amp.GradScaler('cuda', enabled=(DEVICE.type == 'cuda'))

In [15]:
EPOCHS = 10
best_val_acc = 0.0

In [16]:
for epoch in range(EPOCHS):
    model.train()
    running_loss, correct, total = 0.0, 0, 0

    # Training Pass
    for images, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Train]"):
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()

        with torch.cuda.amp.autocast(enabled=(DEVICE.type == 'cuda')):
            outputs = model(images)
            loss = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item() * images.size(0)
        _, preds = torch.max(outputs, 1)
        correct += torch.sum(preds == labels.data)
        total += labels.size(0)

    train_loss = running_loss / total
    train_acc = (correct.double() / total).item()

    # Validation Pass
    model.eval()
    val_loss = 0.0
    val_preds, val_labels = [], []

    with torch.no_grad():
        for images, labels in tqdm(val_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Val]"):
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            
            with torch.amp.autocast('cuda', enabled=(DEVICE.type == 'cuda')):
                outputs = model(images)
                loss = criterion(outputs, labels)

            val_loss += loss.item() * images.size(0)
            _, preds = torch.max(outputs, 1)

            val_preds.extend(preds.cpu().numpy())
            val_labels.extend(labels.cpu().numpy())

    val_loss = val_loss / len(val_dataset)
    val_acc = accuracy_score(val_labels, val_preds)

    print(f"\nEpoch {epoch+1}/{EPOCHS} Summary:")
    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc*100:.2f}%")
    print(f"Val Loss:   {val_loss:.4f} | Val Acc:   {val_acc*100:.2f}%")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), "best_resnet50_dr.pth")
        print("--> Saved best checkpoint: best_resnet50_dr.pth\n")

Epoch 1/10 [Train]:   0%|          | 0/184 [00:00<?, ?it/s]

C:\Users\samar\AppData\Local\Temp\ipykernel_14564\1908411841.py:10: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(DEVICE.type == 'cuda')):


Epoch 1/10 [Val]:   0%|          | 0/23 [00:00<?, ?it/s]


Epoch 1/10 Summary:
Train Loss: 0.8250 | Train Acc: 70.75%
Val Loss:   0.6437 | Val Acc:   78.14%
--> Saved best checkpoint: best_resnet50_dr.pth



Epoch 2/10 [Train]:   0%|          | 0/184 [00:00<?, ?it/s]

...


Epoch 9/10 Summary:
Train Loss: 0.3003 | Train Acc: 88.46%
Val Loss:   0.4570 | Val Acc:   85.79%
--> Saved best checkpoint: best_resnet50_dr.pth



Epoch 10/10 [Train]:   0%|          | 0/184 [00:00<?, ?it/s]

C:\Users\samar\AppData\Local\Temp\ipykernel_14564\1908411841.py:10: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(DEVICE.type == 'cuda')):


Epoch 10/10 [Val]:   0%|          | 0/23 [00:00<?, ?it/s]


Epoch 10/10 Summary:
Train Loss: 0.2919 | Train Acc: 88.87%
Val Loss:   0.4839 | Val Acc:   83.61%


In [17]:
print("--- Detailed Classification Report ---")
print(classification_report(val_labels, val_preds, target_names=[
    "0 - No DR", "1 - Mild", "2 - Moderate", "3 - Severe", "4 - Proliferative"
]))

--- Detailed Classification Report ---
                   precision    recall  f1-score   support

        0 - No DR       0.99      0.99      0.99       172
         1 - Mild       0.73      0.68      0.70        40
     2 - Moderate       0.74      0.86      0.79       104
       3 - Severe       0.20      0.05      0.07        22
4 - Proliferative       0.60      0.64      0.62        28

         accuracy                           0.84       366
        macro avg       0.65      0.64      0.64       366
     weighted avg       0.81      0.84      0.82       366

